# Best vs iHub Audit Viewer

This notebook reads the preserved **2,000-order benchmark CSV**. It does not rerun the benchmark. Use it to separate iHub policy violations, confirm Best has no policy-compliant losses, review all 67 Best wins, then inspect any winning Order ID with the original inputs and validated 3D packing plan.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "notebooks"))

from benchmark_solver_modes import _inputs
from bin_packing_3d import result_tables, solve_order, visualize_result

CSV_PATH = ROOT / "notebooks" / "artifacts" / "benchmark_2000_best_vs_ihub.csv"
DATA_PATH = ROOT / "data" / "raw" / "data_samples_v2.json"

audit = pd.read_csv(CSV_PATH)
records = json.loads(DATA_PATH.read_text(encoding="utf-8"))
records_by_order = {record["input"]["OrderId"]: record for record in records}

display(Markdown(f"## Full benchmark: {len(audit):,} orders"))
display(audit)


In [ ]:
policy_violations = audit[audit["ihub_box9_over_fill_cap"]].copy()
eligible = audit[~audit["ihub_box9_over_fill_cap"]].copy()
wins = eligible[eligible["best_win"]].copy()
losses = eligible[eligible["best_vs_ihub"] == "worse"].copy()

display(Markdown(
    f"## Fair-comparison population\n"
    f"- **{len(policy_violations):,}** iHub results violate the supplied Box9 fill policy and are flagged separately.\n"
    f"- **{len(eligible):,}** orders remain for like-for-like optimization comparison.\n"
    f"- **{len(wins):,}** orders are better with Best.\n"
    f"- **{len(losses):,}** policy-compliant orders are worse with Best."
))

display(Markdown("### iHub policy violations"))
display(policy_violations)

assert len(losses) == 0, "Best has policy-compliant losses. Review before presenting."
assert len(wins) == 67, f"Expected 67 Best wins, found {len(wins)}."

display(Markdown("## The 67 Best wins"))
display(wins[[
    "order_id", "physical_items", "ihub_boxes", "best_boxes",
    "ihub_cartons", "best_cartons",
    "ihub_largest_carton_volume_mm3", "best_largest_carton_volume_mm3",
    "ihub_external_volume_mm3", "best_external_volume_mm3",
    "best_win_reason", "best_runtime_ms", "best_optimality_proven",
]])


## Inspect any of the 67 wins

Set an Order ID from the table above. The helper immediately states the winning criterion, shows the original item dimensions and `VerticalRotation` permission, reruns Best using that order's supplied policy, shows the validated carton/XYZ placement tables, and renders the 3D packing plan.

In [ ]:
def inspect_win(order_id):
    match = wins[wins["order_id"] == order_id]
    if match.empty:
        raise ValueError(f"Order {order_id!r} is not one of the 67 Best wins.")
    row = match.iloc[0]
    record = records_by_order[order_id]
    order, boxes, solver_config = _inputs(record)

    display(Markdown(
        f"# Order {order_id}: {row['best_win_reason']}\n"
        f"**iHub:** {row['ihub_boxes']} | {int(row['ihub_cartons'])} carton(s) | "
        f"largest {int(row['ihub_largest_carton_volume_mm3']):,} mm³ | total {int(row['ihub_external_volume_mm3']):,} mm³  \n"
        f"**Best:** {row['best_boxes']} | {int(row['best_cartons'])} carton(s) | "
        f"largest {int(row['best_largest_carton_volume_mm3']):,} mm³ | total {int(row['best_external_volume_mm3']):,} mm³"
    ))

    item_rows = []
    for item in order["Items"]:
        item_rows.append({
            "Code": item["Code"], "Quantity": item["Quantity"],
            "Length": item["Length"], "Width": item["Width"], "Height": item["Height"],
            "Weight": item["Weight"], "VerticalRotation": item.get("VerticalRotation"),
        })
    display(Markdown("### Original order items and rotation permission"))
    display(pd.DataFrame(item_rows))

    result = solve_order(order, boxes, mode="best", **solver_config)
    if not result.validation.valid:
        raise AssertionError(result.validation.errors)
    summary, placements = result_tables(result)
    display(Markdown("### Best packing result: independently validated"))
    display(summary)
    display(Markdown("### Exact packed orientation and XYZ coordinates"))
    display(placements)
    visualize_result(result)
    return result

# Pick any Order ID from the 67-win table.
INSPECT_ORDER_ID = wins.iloc[0]["order_id"]
result = inspect_win(INSPECT_ORDER_ID)
